# Databricks/PySpark dla Data Analyst/Engineer — Moduł 5: Spark SQL

Do tej pory pisaliśmy wyłącznie w DataFrame API. Ten moduł pokazuje drugą twarz Sparka --
**Spark SQL**: dokładnie te same zapytania SQL, które znasz z kursu SQL/PostgreSQL,
uruchamiane bezpośrednio na naszych DataFrame'ach.

## Spis treści
1. [createOrReplaceTempView i spark.sql()](#sec1)
2. [Mieszanie SQL i DataFrame API](#sec2)
3. [Catalog: przeglądanie zarejestrowanych widoków](#sec3)
4. [Widoki globalne (global temp view)](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. createOrReplaceTempView i spark.sql()

`createOrReplaceTempView("nazwa")` rejestruje DataFrame jako **tymczasowy widok SQL**
(istnieje tylko w ramach bieżącej `SparkSession`, znika po zamknięciu notatnika).
Od tego momentu można na nim pisać zwykłe zapytania SQL przez `spark.sql(...)`.

In [ ]:
klienci.createOrReplaceTempView("klienci_view")
produkty.createOrReplaceTempView("produkty_view")
zamowienia.createOrReplaceTempView("zamowienia_view")
pozycje_zamowien.createOrReplaceTempView("pozycje_zamowien_view")
pracownicy.createOrReplaceTempView("pracownicy_view")

spark.sql("SELECT * FROM klienci_view LIMIT 5").show()


`spark.sql(...)` zwraca zwykły DataFrame -- dokładnie taki sam obiekt, jaki dostalibyśmy
z `select()`/`filter()`. Możesz na nim od razu wywołać dowolną metodę DataFrame API,
albo przypisać go do zmiennej i użyć dalej w kodzie.

In [ ]:
wynik = spark.sql("""
    SELECT segment, COUNT(*) AS liczba_klientow
    FROM klienci_view
    GROUP BY segment
    ORDER BY liczba_klientow DESC
""")
print(type(wynik))
wynik.show()


> ⚡ **Ten sam SQL, co w kursie SQL/PostgreSQL**
>
> Cała składnia poznana w kursie SQL -- `JOIN`, `GROUP BY`, `HAVING`, podzapytania, `CASE WHEN`, funkcje okienkowe -- działa tu niemal identycznie (Spark SQL to dialekt bardzo zbliżony do standardu ANSI SQL, z drobnymi różnicami w niektórych funkcjach specyficznych dla danej bazy danych).

<a id="sec2"></a>
## 2. Mieszanie SQL i DataFrame API

Ponieważ `spark.sql()` zwraca zwykły DataFrame, można swobodnie MIESZAĆ oba style w
jednym potoku -- np. połączyć tabele przez SQL, a dalsze przekształcenia dopisać w
DataFrame API, albo odwrotnie.

In [ ]:
# Join i agregacja w czystym SQL-u
raport_sql = spark.sql("""
    SELECT k.miasto, COUNT(*) AS liczba_zamowien
    FROM zamowienia_view z
    JOIN klienci_view k ON z.klient_id = k.klient_id
    WHERE z.status != 'Anulowane'
    GROUP BY k.miasto
""")

# ...a dalsze sortowanie i limit -- w DataFrame API
raport_sql.orderBy(raport_sql.liczba_zamowien.desc()).limit(3).show()


In [ ]:
from pyspark.sql import functions as F

# Odwrotnie: DataFrame API do przygotowania danych...
przefiltrowane = zamowienia.filter(F.col("status") == "Dostarczone")
przefiltrowane.createOrReplaceTempView("dostarczone_view")

# ...a dalej znowu czysty SQL na nowo zarejestrowanym widoku
spark.sql("SELECT COUNT(*) AS liczba_dostarczonych FROM dostarczone_view").show()


> ⚠️ **Wybór stylu to kwestia czytelności, nie wydajności**
>
> Zarówno SQL, jak i DataFrame API kompilują się do DOKŁADNIE TEGO SAMEGO planu wykonania przez Catalyst -- żaden z nich nie jest szybszy. Wybieraj ten, który w danym miejscu jest czytelniejszy: złożone łączenia i agregacje często wygodniej pisze się w SQL-u, a przekształcenia sterowane zmiennymi Pythona (pętle, warunki) -- w DataFrame API.

<a id="sec3"></a>
## 3. Catalog: przeglądanie zarejestrowanych widoków

`spark.catalog` daje wgląd w to, co aktualnie jest zarejestrowane w bieżącej sesji --
przydatne, żeby sprawdzić, jakie widoki/tabele są dostępne, zanim napiszemy do nich
zapytanie.

In [ ]:
for tabela in spark.catalog.listTables():
    print(tabela.name, "-- tymczasowa:" if tabela.isTemporary else "-- stala:", tabela.tableType)


In [ ]:
for kolumna in spark.catalog.listColumns("klienci_view"):
    print(kolumna.name, kolumna.dataType)


> 🧱 **Unity Catalog w Databricks**
>
> To, co tu widzimy (`spark.catalog`), to najprostszy, lokalny wariant katalogu -- tymczasowe widoki znikające po zamknięciu sesji. W prawdziwym środowisku Databricks **Unity Catalog** rozszerza to o TRWAŁE tabele, uprawnienia dostępu na poziomie kolumn i wierszy, oraz wspólny katalog widoczny dla całej organizacji -- wracamy do tego szerzej w Module 14.

<a id="sec4"></a>
## 4. Widoki globalne (global temp view)

Zwykły `createOrReplaceTempView` jest widoczny tylko w BIEŻĄCEJ sesji Sparka. Jeśli
chcesz udostępnić widok między RÓŻNYMI sesjami w tej samej aplikacji Sparka, użyj
`createOrReplaceGlobalTempView` -- trzeba wtedy odwoływać się do niego przez specjalną
bazę `global_temp`.

In [ ]:
produkty.createOrReplaceGlobalTempView("produkty_globalny")
spark.sql("SELECT * FROM global_temp.produkty_globalny LIMIT 3").show()


> ⚠️ **Global temp view w praktyce -- rzadko potrzebny**
>
> W typowej pracy z pojedynczym notatnikiem (jak w tym kursie) zwykły `createOrReplaceTempView` w zupełności wystarcza. Widoki globalne przydają się głównie przy bardziej złożonych aplikacjach z wieloma niezależnymi `SparkSession` w jednym procesie -- rzadki przypadek w codziennej pracy analityka.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- `createOrReplaceTempView()` -- rejestrowanie DataFrame jako tymczasowego widoku SQL,
- `spark.sql()` -- uruchamianie zapytań SQL, które zwracają zwykły DataFrame,
- swobodne mieszanie stylu SQL i DataFrame API w jednym potoku danych,
- `spark.catalog` -- przeglądanie zarejestrowanych widoków i ich kolumn,
- `createOrReplaceGlobalTempView` -- widoki widoczne między sesjami w tej samej aplikacji.

### 📝 Ćwiczenie 1: Pierwsze zapytanie SQL

Zarejestruj `pracownicy` jako widok `pracownicy_view` i napisz zapytanie SQL zwracające liczbę pracowników w każdym dziale.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
pracownicy.createOrReplaceTempView("pracownicy_view")
spark.sql("""
    SELECT dzial, COUNT(*) AS liczba_pracownikow
    FROM pracownicy_view
    GROUP BY dzial
""").show()
```

</details>

### 📝 Ćwiczenie 2: Join w SQL-u

Napisz zapytanie SQL, które łączy `pozycje_zamowien_view` z `produkty_view` i liczy sumaryczny przychód (`ilosc * cena_jednostkowa`) per kategoria.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
spark.sql("""
    SELECT p.kategoria, ROUND(SUM(poz.ilosc * poz.cena_jednostkowa), 2) AS przychod
    FROM pozycje_zamowien_view poz
    JOIN produkty_view p ON poz.produkt_id = p.produkt_id
    GROUP BY p.kategoria
    ORDER BY przychod DESC
""").show()
```

</details>

### 📝 Ćwiczenie 3: SQL, a potem DataFrame API

Wykonaj zapytanie SQL zwracające `zamowienia_view` przefiltrowane do statusu `'Anulowane'`, a następnie na WYNIKU (już jako DataFrame) wywołaj `.count()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
anulowane = spark.sql("SELECT * FROM zamowienia_view WHERE status = 'Anulowane'")
print(f"Liczba anulowanych: {anulowane.count()}")
```

</details>

### 📝 Ćwiczenie 4: Sprawdzenie katalogu

Wypisz przez `spark.catalog.listTables()` nazwy WSZYSTKICH widoków zarejestrowanych do tej pory w tym notatniku.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
for t in spark.catalog.listTables():
    print(t.name)
```

</details>

> 🔥 **Wyzwanie: podzapytanie i CASE WHEN w Spark SQL**
>
> Napisz JEDNO zapytanie SQL (z podzapytaniem, jak w Module 6 kursu SQL), które: dla każdego klienta liczy sumę wartości jego zamówień (`ilosc * cena_jednostkowa * (1 - rabat)`), a następnie przez `CASE WHEN` kategoryzuje klientów na `'VIP'` (suma > 5000) i `'Zwykly'` -- policz, ilu klientów wpada do każdej kategorii.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
spark.sql("""
    SELECT
        CASE WHEN suma_wartosci > 5000 THEN 'VIP' ELSE 'Zwykly' END AS kategoria,
        COUNT(*) AS liczba_klientow
    FROM (
        SELECT
            z.klient_id,
            SUM(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat)) AS suma_wartosci
        FROM zamowienia_view z
        JOIN pozycje_zamowien_view poz ON z.zamowienie_id = poz.zamowienie_id
        GROUP BY z.klient_id
    ) AS suma_per_klient
    GROUP BY kategoria
""").show()
```

To dokladnie ten sam wzorzec podzapytania w klauzuli FROM, ktory poznalismy w Module 6 kursu SQL -- w Spark SQL dziala identycznie.

</details>

## Co dalej?

W **Module 6** zajmiemy się wczytywaniem i zapisywaniem danych: pliki CSV/JSON/Parquet,
schemat jawny kontra wnioskowany, oraz partycjonowanie przy zapisie.